# SE3082 - Parallel Computing
## Lab Sheet 10 - CUDA (Google Colab Edition)
Student ID: IT24610789


## Colab Set-up
Check that the GPU is visible and find compute capability.


In [ ]:
!nvidia-smi
!nvidia-smi --query-gpu=name,compute_cap --format=csv


## Exercise 1 - Timing a GPU program
### Part (a) - Timed 512-block vector addition


In [ ]:
%%writefile ex1_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);
    double dStart, dElaps;

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size);
    random_ints(a, N);
    b = (int *)malloc(size);
    random_ints(b, N);
    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    dStart = cpuSecond();

    add<<<N, 1>>>(d_a, d_b, d_c);
    cudaDeviceSynchronize();

    dElaps = cpuSecond() - dStart;

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < 5; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    free(a);
    free(b);
    free(c);
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    return 0;
}


In [ ]:
!nvcc -arch=sm_75 ex1_timed.cu -o ex1_timed && ./ex1_timed


### Part (b) - Time your Lab 9 programs
#### Timed Exercise 3 (512 blocks x 1 thread)


In [ ]:
%%writefile ex3_blocks_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);
    double dStart, dElaps;

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size);
    random_ints(a, N);
    b = (int *)malloc(size);
    random_ints(b, N);
    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    dStart = cpuSecond();
    add<<<N, 1>>>(d_a, d_b, d_c);
    cudaDeviceSynchronize();
    dElaps = cpuSecond() - dStart;
    printf("Time taken - %.7f\n", dElaps);

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    free(a);
    free(b);
    free(c);
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    return 0;
}


In [ ]:
!nvcc -arch=sm_75 ex3_blocks_timed.cu -o ex3_blocks_timed && ./ex3_blocks_timed


#### Timed Exercise 4 (1 block x 512 threads)


In [ ]:
%%writefile ex4_threads_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

__global__ void addT(int *a, int *b, int *c) {
    c[threadIdx.x] = a[threadIdx.x] + b[threadIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);
    double dStart, dElaps;

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size);
    random_ints(a, N);
    b = (int *)malloc(size);
    random_ints(b, N);
    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    dStart = cpuSecond();
    addT<<<1, N>>>(d_a, d_b, d_c);
    cudaDeviceSynchronize();
    dElaps = cpuSecond() - dStart;
    printf("Time taken - %.7f\n", dElaps);

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    free(a);
    free(b);
    free(c);
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    return 0;
}


In [ ]:
!nvcc -arch=sm_75 ex4_threads_timed.cu -o ex4_threads_timed && ./ex4_threads_timed


#### Timed Exercise 5 (10,000,000 elements, 19,532 blocks x 512 threads)


In [ ]:
%%writefile ex5_vecmul_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 10000000
#define THREADS_PER_BLOCK 512

__global__ void vecMul(int *a, int *b, int *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        c[i] = a[i] * b[i];
    }
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    size_t size = (size_t)N * sizeof(int);
    double dStart, dElaps;

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size);
    random_ints(a, N);
    b = (int *)malloc(size);
    random_ints(b, N);
    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;

    dStart = cpuSecond();
    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);
    cudaDeviceSynchronize();
    dElaps = cpuSecond() - dStart;
    printf("Time taken - %.7f\n", dElaps);

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] * b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);

    free(a);
    free(b);
    free(c);
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    return 0;
}


In [ ]:
!nvcc -arch=sm_75 ex5_vecmul_timed.cu -o ex5_vecmul_timed && ./ex5_vecmul_timed


#### Timed Exercise 6 (100,000,000 elements, 313 x 625 blocks of 32 x 16)


In [ ]:
%%writefile ex6_matrix_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define ROWS 10000
#define COLS 10000

__global__ void mulElem(int *a, int *b, int *c, int rows, int cols) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < rows && col < cols) {
        int idx = row * cols + col;
        c[idx] = a[idx] * b[idx];
    }
}

void random_ints(int *x, size_t size) {
    for (size_t i = 0; i < size; i++)
        x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    size_t count = (size_t)ROWS * COLS;
    size_t size = count * sizeof(int);
    double dStart, dElaps;

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size);
    random_ints(a, count);
    b = (int *)malloc(size);
    random_ints(b, count);
    c = (int *)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    dim3 threadsPerBlock(32, 16);
    dim3 numBlocks((COLS + threadsPerBlock.x - 1) / threadsPerBlock.x, (ROWS + threadsPerBlock.y - 1) / threadsPerBlock.y);

    dStart = cpuSecond();
    mulElem<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, ROWS, COLS);
    cudaDeviceSynchronize();
    dElaps = cpuSecond() - dStart;
    printf("Time taken - %.7f\n", dElaps);

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    size_t errors = 0;
    for (size_t i = 0; i < count; i++) {
        if (c[i] != a[i] * b[i]) errors++;
    }
    printf("Verification: %zu mismatches out of %zu\n", errors, count);

    free(a);
    free(b);
    free(c);
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    return 0;
}


In [ ]:
!nvcc -arch=sm_75 ex6_matrix_timed.cu -o ex6_matrix_timed && ./ex6_matrix_timed


### Part (c) - Kernel time versus copy time (100,000,000 elements)


In [ ]:
%%writefile ex1_part_c.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 100000000
#define THREADS_PER_BLOCK 512

__global__ void vecMul(int *a, int *b, int *c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) {
        c[i] = a[i] * b[i];
    }
}

void random_ints(int *x, size_t size) {
    for (size_t i = 0; i < size; i++)
        x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    size_t size = (size_t)N * sizeof(int);
    double dStartWhole, dElapsWhole;
    double dStartKernel, dElapsKernel;

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int *)malloc(size);
    random_ints(a, N);
    b = (int *)malloc(size);
    random_ints(b, N);
    c = (int *)malloc(size);

    int blocks = (N + THREADS_PER_BLOCK - 1) / THREADS_PER_BLOCK;

    dStartWhole = cpuSecond();

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    dStartKernel = cpuSecond();
    vecMul<<<blocks, THREADS_PER_BLOCK>>>(d_a, d_b, d_c, N);
    cudaDeviceSynchronize();
    dElapsKernel = cpuSecond() - dStartKernel;

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    dElapsWhole = cpuSecond() - dStartWhole;

    printf("Kernel only time taken - %.7f\n", dElapsKernel);
    printf("Whole sequence time taken - %.7f\n", dElapsWhole);

    free(a);
    free(b);
    free(c);
    cudaFree(d_a);
    cudaFree(d_b);
    cudaFree(d_c);
    return 0;
}


In [ ]:
!nvcc -arch=sm_75 ex1_part_c.cu -o ex1_part_c && ./ex1_part_c


## Exercise 2 - Catching errors with the CHECK() macro


In [ ]:
%%writefile ex2_check.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

#define CHECK(call)                                                          \
    do {                                                                     \
        const cudaError_t error = (call);                                    \
        if (error != cudaSuccess) {                                          \
            printf("Error: %s:%d, ", __FILE__, __LINE__);                    \
            printf("code:%d, reason: %s\n", error, cudaGetErrorString(error));\
            exit(1);                                                         \
        }                                                                    \
    } while (0)

__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);
    double dStart, dElaps;

    CHECK(cudaMalloc((void **)&d_a, size));
    CHECK(cudaMalloc((void **)&d_b, size));
    CHECK(cudaMalloc((void **)&d_c, size));

    a = (int *)malloc(size);
    random_ints(a, N);
    b = (int *)malloc(size);
    random_ints(b, N);
    c = (int *)malloc(size);

    CHECK(cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice));

    dStart = cpuSecond();

    add<<<N, 1>>>(d_a, d_b, d_c);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());

    dElaps = cpuSecond() - dStart;

    CHECK(cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost));

    for (int r = 0; r < 5; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken - %.7f\n", dElaps);

    free(a);
    free(b);
    free(c);
    CHECK(cudaFree(d_a));
    CHECK(cudaFree(d_b));
    CHECK(cudaFree(d_c));
    return 0;
}


In [ ]:
!nvcc -arch=sm_75 ex2_check.cu -o ex2_check && ./ex2_check


## Exercise 3 - Scaling a picture with a 2D grid (PictureKernel)


In [ ]:
%%writefile ex3_picture.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define WIDTH 800
#define HEIGHT 600

#define CHECK(call)                                                          \
    do {                                                                     \
        const cudaError_t error = (call);                                    \
        if (error != cudaSuccess) {                                          \
            printf("Error: %s:%d, ", __FILE__, __LINE__);                    \
            printf("code:%d, reason: %s\n", error, cudaGetErrorString(error));\
            exit(1);                                                         \
        }                                                                    \
    } while (0)

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

__global__ void PictureKernel(unsigned char *d_Pin, unsigned char *d_Pout,
                              int height, int width) {
    int Row = blockIdx.y * blockDim.y + threadIdx.y;
    int Col = blockIdx.x * blockDim.x + threadIdx.x;

    if ((Row < height) && (Col < width)) {
        int v = 2 * d_Pin[Row * width + Col];
        d_Pout[Row * width + Col] = (v > 255) ? 255 : v;
    }
}

int main(void) {
    size_t count = (size_t)WIDTH * HEIGHT;
    size_t size = count * sizeof(unsigned char);
    unsigned char *h_in, *h_out;
    unsigned char *d_in, *d_out;
    double dStart, dElaps;

    h_in = (unsigned char *)malloc(size);
    h_out = (unsigned char *)malloc(size);

    CHECK(cudaMalloc((void **)&d_in, size));
    CHECK(cudaMalloc((void **)&d_out, size));

    for (size_t i = 0; i < count; i++) {
        h_in[i] = (unsigned char)(rand() % 256);
    }

    CHECK(cudaMemcpy(d_in, h_in, size, cudaMemcpyHostToDevice));

    dim3 threadsPerBlock(16, 16);
    dim3 numBlocks((WIDTH + threadsPerBlock.x - 1) / threadsPerBlock.x,
                   (HEIGHT + threadsPerBlock.y - 1) / threadsPerBlock.y);

    printf("Grid: %d x %d blocks of %d x %d threads\n",
           numBlocks.x, numBlocks.y, threadsPerBlock.x, threadsPerBlock.y);

    dStart = cpuSecond();

    PictureKernel<<<numBlocks, threadsPerBlock>>>(d_in, d_out, HEIGHT, WIDTH);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());

    dElaps = cpuSecond() - dStart;

    CHECK(cudaMemcpy(h_out, d_out, size, cudaMemcpyDeviceToHost));

    printf("Original 50x50 portion:\n");
    for (int r = 0; r < 50; r++) {
        for (int c = 0; c < 50; c++) {
            printf("%3d ", h_in[r * WIDTH + c]);
        }
        printf("\n");
    }

    printf("Updated 50x50 portion:\n");
    for (int r = 0; r < 50; r++) {
        for (int c = 0; c < 50; c++) {
            printf("%3d ", h_out[r * WIDTH + c]);
        }
        printf("\n");
    }

    int mismatches = 0;
    int capped = 0;
    for (size_t i = 0; i < count; i++) {
        int expected = 2 * h_in[i];
        if (expected > 255) {
            expected = 255;
            capped++;
        }
        if (h_out[i] != expected) {
            mismatches++;
        }
    }

    printf("Verification: %d mismatches out of %zu\n", mismatches, count);
    printf("Pixels capped at 255: %d\n", capped);
    printf("Time taken - %.7f\n", dElaps);

    free(h_in);
    free(h_out);
    CHECK(cudaFree(d_in));
    CHECK(cudaFree(d_out));
    return 0;
}


In [ ]:
!nvcc -arch=sm_75 ex3_picture.cu -o ex3_picture && ./ex3_picture


## Exercise 4 - Multiplying two 3D matrices element by element (3D grid)


In [ ]:
%%writefile ex4_volume.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define NX 500
#define NY 500
#define NZ 500

#define CHECK(call)                                                          \
    do {                                                                     \
        const cudaError_t error = (call);                                    \
        if (error != cudaSuccess) {                                          \
            printf("Error: %s:%d, ", __FILE__, __LINE__);                    \
            printf("code:%d, reason: %s\n", error, cudaGetErrorString(error));\
            exit(1);                                                         \
        }                                                                    \
    } while (0)

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

__global__ void mulElem3D(int *a, int *b, int *c, int nx, int ny, int nz) {
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    int z = blockIdx.z * blockDim.z + threadIdx.z;

    if (x < nx && y < ny && z < nz) {
        size_t idx = ((size_t)z * ny + y) * nx + x;
        c[idx] = a[idx] * b[idx];
    }
}

void random_ints(int *x, size_t size) {
    for (size_t i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    size_t count = (size_t)NX * NY * NZ;
    size_t size = count * sizeof(int);
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    double dStart, dElaps;

    CHECK(cudaMalloc((void **)&d_a, size));
    CHECK(cudaMalloc((void **)&d_b, size));
    CHECK(cudaMalloc((void **)&d_c, size));

    a = (int *)malloc(size);
    random_ints(a, count);
    b = (int *)malloc(size);
    random_ints(b, count);
    c = (int *)malloc(size);

    CHECK(cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice));

    dim3 threadsPerBlock(8, 8, 8);
    dim3 numBlocks((NX + threadsPerBlock.x - 1) / threadsPerBlock.x,
                   (NY + threadsPerBlock.y - 1) / threadsPerBlock.y,
                   (NZ + threadsPerBlock.z - 1) / threadsPerBlock.z);

    printf("Grid: %d x %d x %d blocks\n", numBlocks.x, numBlocks.y, numBlocks.z);

    dStart = cpuSecond();

    mulElem3D<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, NX, NY, NZ);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());

    dElaps = cpuSecond() - dStart;

    CHECK(cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost));

    for (size_t idx = count - 1000; idx < count; idx++) {
        int cur_z = idx / ((size_t)NY * NX);
        int rem = idx % ((size_t)NY * NX);
        int cur_y = rem / NX;
        int cur_x = rem % NX;
        printf("C[%d][%d][%d] = %d x %d = %d\n", cur_z, cur_y, cur_x, a[idx], b[idx], c[idx]);
    }

    size_t errors = 0;
    for (size_t i = 0; i < count; i++) {
        if (c[i] != a[i] * b[i]) errors++;
    }
    printf("Verification: %zu mismatches out of %zu\n", errors, count);
    printf("Time taken - %.7f\n", dElaps);

    free(a);
    free(b);
    free(c);
    CHECK(cudaFree(d_a));
    CHECK(cudaFree(d_b));
    CHECK(cudaFree(d_c));
    return 0;
}


In [ ]:
!nvcc -arch=sm_75 ex4_volume.cu -o ex4_volume && ./ex4_volume
